# **Add Split — create an immutable split folder (per-dataset files, unified quad schema)**

*   INPUT: preprocessed benchmark CSV (post-AF) + external coco CSV
*   OUTPUT: one **split folder** under `splits/<SPLIT_NAME>/` — one file per dataset per split,
    all sharing the unified quad schema.

**Unified schema** — one row = one comparison, want `cos(source_a, close_a) > cos(source_b, far_b)`:

| column | meaning |
|---|---|
| `source_a`, `close_a` | the pair that **should win** |
| `source_b`, `far_b`   | the pair that **should lose** |
| `category` | `regular` / `easy_close` / `easy_far` / `zigzag` / `coco` |
| `group` |

⚠ Column names are **roles, not text types**: in an `easy_far` row the `close_a` slot holds a
far text (the should-win pair), and `far_b` may hold another source's close. Triplets are the
special case `source_a == source_b`.

```
splits/<SPLIT_NAME>/
├── split_meta.json                fingerprint + full provenance
├── regular_train/val/test.csv    benchmark triplets as quads (group filled)
├── easy_close_train/val/test.csv
├── easy_far_train/val/test.csv
├── zigzag_train/val/test.csv
├── coco_train/val/test.csv       train/val sampled per PCT; test = FULL coco_test
├── coco_split.csv                aux: coco image_id → split (for later resampling)
└── finetuned_models/             training notebook saves models HERE
```

**One file per dataset per split** — a training run composes its mix by concatenating files
(control = `regular_train.csv`; full = all five; no-coco = all except coco). All 15 files are
written even when empty, so downstream existence-asserts never break.

* A split folder is **immutable once created** — this notebook refuses to overwrite one that
  already has trained models inside.


In [ ]:
# Hyper Parameters — everything tunable lives here

# --- INPUT ---
DRIVE_ROOT = '/content/drive/MyDrive/PROJECT_DATA'
DATA_PATH  = 'DataFolder_08.07'
BENCHMARK_CSV_PATH = f'{DRIVE_ROOT}/{DATA_PATH}/benchmark_cleaned.csv'   # post-AF triplets (+ all metadata cols)
COCO_CSV_PATH      = f'{DRIVE_ROOT}/{DATA_PATH}/coco_triplets.csv'  # external cross-image set

# --- OUTPUT ---
SPLITS_ROOT = f'{DRIVE_ROOT}/{DATA_PATH}/splits'
SPLIT_NAME  = 'Split_2'      # None = auto-derive from characteristics, e.g.

# --- Split fractions (source-level for benchmark; image-id-level for coco) ---
TRAIN_FRAC, VAL_FRAC = 0.70, 0.15          # test = the rest
COCO_TRAIN_FRAC, COCO_VAL_FRAC = 0.7, 0.15

# --- Quad augmentation (fractions of each split's regular-triplet count) ---
PCT_EASY_CLOSE = 0.03
PCT_EASY_FAR   = 0.03
PCT_ZIGZAG     = 0.03
PCT_COCO       = 0.10   # coco quads mixed into TRAIN/VAL only (test coco stays separate)

# --- Metadata ---
# ALL benchmark columns beyond (source, close, far) ride along with each quad.
# Naming is by side: *_close describe the close text, *_far describe the far text.
#   regular : close_a from (source,close), far_b from (source,far), same triplet
#             -> gets close-side + far-side + triplet-level cols (triplet_id, image_id)
#   zigzag  : close_a from source A, far_b from source B (different triplets)
#             -> *_close from (source_a,close_a); *_far from (source_b,far_b); no triplet-level
#   easy_close / easy_far / coco : no metadata (blank)

SEED = 42
print("Hyperparameters set.")


Hyperparameters set.


# **Section 1: Utilities & load**

In [ ]:
# Section 1.1 — Imports, Drive mount, load inputs

import os, json, hashlib
from datetime import datetime

import numpy as np
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')
np.random.seed(SEED)

assert os.path.exists(BENCHMARK_CSV_PATH), f"benchmark not found: {BENCHMARK_CSV_PATH}"
bench = pd.read_csv(BENCHMARK_CSV_PATH)
assert {'source', 'close', 'far'}.issubset(bench.columns), \
    f"expected (source, close, far), got {list(bench.columns)}"
META_COLS    = [c for c in bench.columns if c not in ('source', 'close', 'far')]
CLOSE_META   = [c for c in META_COLS if c.endswith('_close')]   # describe close_a
FAR_META     = [c for c in META_COLS if c.endswith('_far')]     # describe far_b
TRIPLET_META = [c for c in META_COLS if c not in CLOSE_META + FAR_META]  # triplet_id, image_id
print(f"Benchmark: {len(bench):,} triplets, {bench['source'].nunique()} sources")
print(f"  close-side meta : {CLOSE_META}")
print(f"  far-side meta   : {FAR_META}")
print(f"  triplet-level   : {TRIPLET_META}  (regular rows only)")

assert os.path.exists(COCO_CSV_PATH), f"coco set not found: {COCO_CSV_PATH}"
coco = pd.read_csv(COCO_CSV_PATH)
coco = coco.dropna(subset=['source', 'close', 'far']) \
           .drop_duplicates(subset=['source', 'close', 'far']).reset_index(drop=True)
_coco_key = 'source_image_id' if 'source_image_id' in coco.columns else 'source'
print(f"Coco: {len(coco):,} triplets (split key: {_coco_key})")


# **Section 2: Split**

In [ ]:
# Section 2.1 — Source-level 70/15/15 (verified) + coco split by image id

def verify_source_split(splits, full_sources, verbose=True):
    names = list(splits); sets = {k: set(v) for k, v in splits.items()}
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            ov = sets[names[i]] & sets[names[j]]
            assert not ov, f"SOURCE LEAK {names[i]}&{names[j]}: {len(ov)} shared, e.g. {list(ov)[:3]}"
    union = set().union(*sets.values()); full = set(full_sources)
    assert not (full - union) and not (union - full), "split does not cover sources exactly"
    if verbose:
        for k in names: print(f"  {k:5s}: {len(sets[k])} sources")
        print("  VERIFIED: disjoint by source, full coverage. No leakage.")

all_sources = sorted(bench['source'].unique().tolist())
rng = np.random.RandomState(SEED); rng.shuffle(all_sources)
n = len(all_sources); n_tr = int(TRAIN_FRAC * n); n_va = int(VAL_FRAC * n)
train_sources = sorted(all_sources[:n_tr])
val_sources   = sorted(all_sources[n_tr:n_tr + n_va])
test_sources  = sorted(all_sources[n_tr + n_va:])

trip_train = bench[bench['source'].isin(train_sources)].reset_index(drop=True)
trip_val   = bench[bench['source'].isin(val_sources)].reset_index(drop=True)
trip_test  = bench[bench['source'].isin(test_sources)].reset_index(drop=True)
verify_source_split({'train': trip_train['source'].unique(),
                     'val':   trip_val['source'].unique(),
                     'test':  trip_test['source'].unique()}, all_sources)
print(f"Triplets: {len(trip_train):,} train / {len(trip_val):,} val / {len(trip_test):,} test")

_ids = sorted(coco[_coco_key].unique().tolist())
_rng = np.random.RandomState(SEED); _rng.shuffle(_ids)
_n = len(_ids); _ntr = int(COCO_TRAIN_FRAC * _n); _nva = int(COCO_VAL_FRAC * _n)
coco_train = coco[coco[_coco_key].isin(set(_ids[:_ntr]))].reset_index(drop=True)
coco_val   = coco[coco[_coco_key].isin(set(_ids[_ntr:_ntr + _nva]))].reset_index(drop=True)
coco_test  = coco[coco[_coco_key].isin(set(_ids[_ntr + _nva:]))].reset_index(drop=True)
assert not (set(coco_train[_coco_key]) & set(coco_test[_coco_key])), "coco split leak"
assert not (set(coco_train[_coco_key]) & set(coco_val[_coco_key])), "coco split leak"
print(f"Coco:     {len(coco_train):,} train / {len(coco_val):,} val / {len(coco_test):,} test")


# **Section 3: Build quads (unified schema)**

In [ ]:
# Section 3.1 — build_quads (dedup + top-up; zigzag both directions), coco quads, unify

def build_quads(df_split, pct_easy_close, pct_easy_far, pct_zigzag, seed):
    """One split's (source, close, far) triplets -> quads (a1,b1,a2,b2,category).
    Negatives sampled ONLY within this split."""
    rng = np.random.RandomState(seed)
    df_split = df_split.reset_index(drop=True)
    regular = pd.DataFrame({
        'a1': df_split['source'], 'b1': df_split['close'],
        'a2': df_split['source'], 'b2': df_split['far'], 'category': 'regular'})
    N = len(regular)

    src_close = {s: sorted(set(v)) for s, v in df_split.groupby('source')['close']}
    src_far   = {s: sorted(set(v)) for s, v in df_split.groupby('source')['far']}
    sources   = sorted(src_close.keys())
    if len(sources) < 2:
        return regular
    close_rows = [(s, c) for s in sources for c in src_close[s]]
    far_rows   = [(s, f) for s in sources for f in src_far[s]]

    def other(a):
        b = a
        while b == a: b = sources[rng.randint(len(sources))]
        return b
    def pick(lst): return lst[rng.randint(len(lst))]

    def sample_easy(pos_rows, target, category):
        seen, rows, attempts = set(), [], 0
        while len(rows) < target and attempts < 20 * max(target, 1):
            attempts += 1
            a, P = pick(pos_rows); b = other(a)
            Xb = pick(src_close[b] + src_far[b])
            key = (a, P, a, Xb)
            if key in seen: continue
            seen.add(key)
            rows.append({'a1': a, 'b1': P, 'a2': a, 'b2': Xb, 'category': category})
        return rows

    def sample_zigzag(target):
        seen, rows, attempts = set(), [], 0
        while len(rows) < target and attempts < 20 * max(target, 1):
            attempts += 1
            a = sources[rng.randint(len(sources))]; b = other(a)
            for key in [(a, pick(src_close[a]), b, pick(src_far[b])),
                        (b, pick(src_close[b]), a, pick(src_far[a]))]:
                if len(rows) >= target or key in seen: continue
                seen.add(key)
                rows.append({'a1': key[0], 'b1': key[1], 'a2': key[2], 'b2': key[3],
                             'category': 'zigzag'})
        return rows

    frames = [regular]
    for rows in [sample_easy(close_rows, int(pct_easy_close * N), 'easy_close'),
                 sample_easy(far_rows,   int(pct_easy_far   * N), 'easy_far'),
                 sample_zigzag(int(pct_zigzag * N))]:
        if rows: frames.append(pd.DataFrame(rows))
    return (pd.concat(frames, ignore_index=True)
              .drop_duplicates(subset=['a1', 'b1', 'a2', 'b2'])
              .sample(frac=1, random_state=seed).reset_index(drop=True))


def coco_quads(pool, target, seed):
    take = pool.sample(n=min(target, len(pool)), random_state=seed).reset_index(drop=True)
    return pd.DataFrame({'a1': take['source'], 'b1': take['close'],
                         'a2': take['source'], 'b2': take['far'], 'category': 'coco'})


quads_train = build_quads(trip_train, PCT_EASY_CLOSE, PCT_EASY_FAR, PCT_ZIGZAG, SEED)
quads_val   = build_quads(trip_val,   PCT_EASY_CLOSE, PCT_EASY_FAR, PCT_ZIGZAG, SEED + 1)
quads_test  = build_quads(trip_test,  PCT_EASY_CLOSE, PCT_EASY_FAR, PCT_ZIGZAG, SEED + 2)

# coco: SAMPLED into train/val (PCT_COCO); FULL coco_test into test (the eval column)
quads_train = pd.concat([quads_train,
    coco_quads(coco_train, int(PCT_COCO * len(trip_train)), SEED)], ignore_index=True)
quads_val = pd.concat([quads_val,
    coco_quads(coco_val, int(PCT_COCO * len(trip_val)), SEED + 1)], ignore_index=True)
quads_test = pd.concat([quads_test,
    coco_quads(coco_test, len(coco_test), SEED + 2)], ignore_index=True)


UNIFIED_COLS = ['source_a', 'close_a', 'source_b', 'far_b', 'category'] + META_COLS

def to_unified(q, trip_split):
    """Unified schema + carry benchmark metadata by SIDE.
      close-side (*_close): keyed on (source_a, close_a) -> the close text's own triplet
      far-side   (*_far)  : keyed on (source_b, far_b)   -> the far text's own triplet
    Applied to {regular, zigzag}. For regular both keys hit the same triplet; for zigzag
    they hit source A and source B respectively. triplet-level cols (triplet_id, image_id)
    only make sense for regular (a single triplet). easy_close/easy_far/coco stay blank."""
    u = q.rename(columns={'a1': 'source_a', 'b1': 'close_a',
                          'a2': 'source_b', 'b2': 'far_b'})
    for c in META_COLS:
        u[c] = ''
    has_meta = u['category'].isin(['regular', 'zigzag'])
    if has_meta.any() and META_COLS and 'source' in trip_split.columns:
        cmap = trip_split.drop_duplicates(['source', 'close']).set_index(['source', 'close'])
        fmap = trip_split.drop_duplicates(['source', 'far']).set_index(['source', 'far'])
        ck = list(zip(u.loc[has_meta, 'source_a'], u.loc[has_meta, 'close_a']))
        fk = list(zip(u.loc[has_meta, 'source_b'], u.loc[has_meta, 'far_b']))
        for c in CLOSE_META:
            u.loc[has_meta, c] = cmap[c].reindex(ck).to_numpy()
        for c in FAR_META:
            u.loc[has_meta, c] = fmap[c].reindex(fk).to_numpy()
        reg = u['category'] == 'regular'
        if reg.any() and TRIPLET_META:
            tmap = (trip_split.drop_duplicates(['source', 'close', 'far'])
                              .set_index(['source', 'close', 'far']))
            tk = list(zip(u.loc[reg, 'source_a'], u.loc[reg, 'close_a'], u.loc[reg, 'far_b']))
            for c in TRIPLET_META:
                u.loc[reg, c] = tmap[c].reindex(tk).to_numpy()
    return (u[UNIFIED_COLS]
            .sample(frac=1, random_state=SEED).reset_index(drop=True))

train_df = to_unified(quads_train, trip_train)
val_df   = to_unified(quads_val,   trip_val)
test_df  = to_unified(quads_test,  trip_test)

for nm, u in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f"{nm:5s}: {len(u):,}  {u['category'].value_counts().to_dict()}")

# --- Leakage guard: prove every quad's sources (except external coco) live in that
#     split's source set, and the three splits stay source-disjoint after augmentation ---
_srcset = {'train': set(train_sources), 'val': set(val_sources), 'test': set(test_sources)}
_used = {}
for nm, u in [('train', train_df), ('val', val_df), ('test', test_df)]:
    nc = u[u['category'] != 'coco']
    used = set(nc['source_a']) | set(nc['source_b'])
    stray = used - _srcset[nm]
    assert not stray, f"{nm}: {len(stray)} source(s) leaked into quads, e.g. {list(stray)[:3]}"
    _used[nm] = used
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    ov = _used[a] & _used[b]
    assert not ov, f"SOURCE LEAK between {a} & {b} quads: {len(ov)} shared, e.g. {list(ov)[:3]}"
print("Quad source-leakage check: PASS "
      "(regular/zigzag/easy_close/easy_far all within-split; splits disjoint)")


# **Section 4: Write the split folder (+ report)**

In [ ]:
# Section 4.1 — Derive name, refuse unsafe overwrites, write everything + meta

def _fingerprint():
    """Stable hash of what defines this split: test sources + coco test ids + sizes."""
    h = hashlib.sha256()
    h.update('|'.join(test_sources).encode())
    h.update('|'.join(map(str, sorted(coco_test[_coco_key].astype(str)))).encode())
    h.update(f'{len(bench)}|{len(coco)}|{SEED}'.encode())
    return h.hexdigest()[:16]

def _derive_name():
    pct = lambda x: f"{int(round(x * 100))}"
    return (f"split_{len(bench)//1000}k_"
            f"{int(TRAIN_FRAC*100)}-{int(VAL_FRAC*100)}-{int(round((1-TRAIN_FRAC-VAL_FRAC)*100))}_"
            f"ec{pct(PCT_EASY_CLOSE)}-ef{pct(PCT_EASY_FAR)}-zz{pct(PCT_ZIGZAG)}-cc{pct(PCT_COCO)}_"
            f"s{SEED}")

split_name = SPLIT_NAME or _derive_name()
split_dir = f'{SPLITS_ROOT}/{split_name}'

if os.path.exists(f'{split_dir}/split_meta.json'):
    models_inside = os.path.isdir(f'{split_dir}/finetuned_models') and \
                    len(os.listdir(f'{split_dir}/finetuned_models')) > 0
    raise RuntimeError(
        f"Split folder already exists: {split_dir}\n"
        + ("It contains trained models — IMMUTABLE, will not overwrite. "
           "Choose a new SPLIT_NAME." if models_inside else
           "Delete it manually or choose a new SPLIT_NAME if you really want to recreate it."))

os.makedirs(f'{split_dir}/finetuned_models', exist_ok=True)

# --- Write: one file per dataset per split; ALL categories written even when empty ---
ALL_CATS = ['regular', 'easy_close', 'easy_far', 'zigzag', 'coco']
# UNIFIED_COLS defined in Section 3.1 (includes carried metadata columns)
files = {}
for split_lbl, u in [('train', train_df), ('val', val_df), ('test', test_df)]:
    for cat in ALL_CATS:
        sub = u[u['category'] == cat].reset_index(drop=True)
        files[f'{cat}_{split_lbl}.csv'] = sub[UNIFIED_COLS] if len(sub) else \
            pd.DataFrame(columns=UNIFIED_COLS)
for fname, df in sorted(files.items()):
    df.to_csv(f'{split_dir}/{fname}', index=False)

# Aux: coco image_id -> split membership (enables later resampling at a different PCT)
coco_split = pd.concat([
    pd.DataFrame({_coco_key: coco_train[_coco_key].unique(), 'split': 'train'}),
    pd.DataFrame({_coco_key: coco_val[_coco_key].unique(),   'split': 'val'}),
    pd.DataFrame({_coco_key: coco_test[_coco_key].unique(),  'split': 'test'}),
], ignore_index=True)
coco_split.to_csv(f'{split_dir}/coco_split.csv', index=False)

meta = {
    'split_name': split_name,
    'fingerprint': _fingerprint(),
    'created_at': datetime.now().isoformat(),
    'inputs': {'benchmark_csv': BENCHMARK_CSV_PATH, 'n_benchmark': len(bench),
               'coco_csv': COCO_CSV_PATH, 'n_coco': len(coco), 'coco_split_key': _coco_key},
    'seed': SEED,
    'fractions': {'train': TRAIN_FRAC, 'val': VAL_FRAC,
                  'coco_train': COCO_TRAIN_FRAC, 'coco_val': COCO_VAL_FRAC},
    'pcts': {'easy_close': PCT_EASY_CLOSE, 'easy_far': PCT_EASY_FAR,
             'zigzag': PCT_ZIGZAG, 'coco': PCT_COCO},
    'sources': {'train': len(train_sources), 'val': len(val_sources), 'test': len(test_sources)},
    'counts': {fname: len(df) for fname, df in files.items()},
    'categories': {nm: u['category'].value_counts().to_dict()
                   for nm, u in [('train', train_df), ('val', val_df), ('test', test_df)]},
    'format': UNIFIED_COLS,
    'meta_cols': META_COLS,
}
with open(f'{split_dir}/split_meta.json', 'w') as fh:
    json.dump(meta, fh, indent=2)

print(f"{'='*70}\nSPLIT CREATED: {split_name}\n{'='*70}")
print(f"Folder:      {split_dir}")
print(f"Fingerprint: {meta['fingerprint']}")
print(f"Sources:     {len(train_sources)} train / {len(val_sources)} val / {len(test_sources)} test")
print(f"\nFiles written:")
for fname, df in sorted(files.items()):
    print(f"  {fname:26s} {len(df):>10,} rows")
print(f"  coco_split.csv             {len(coco_split):>10,} rows (image_id -> split)")
print(f"  split_meta.json")
print(f"  finetuned_models/          (empty — training notebook writes here)")
print(f"\nNext: point Finetune.ipynb's SPLIT_DIR at this folder.")
